<a href="https://colab.research.google.com/github/zhannaa0/kazakh-hybrid-nlp/blob/main/notebooks/02_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 2. Методы и модели (методичка 3.12)

Rule-based → нейросеть → гибрид → сравнение 3×3. Описание методов, формулы, допущения —
`docs/methods.md`.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (нужен для шага 2).
Сначала должен быть выполнен этап 1 (`01_stage1_data.ipynb`): нужны `thesis/data/processed/`.

Все результаты пишутся на Drive (`thesis/outputs/`), и каждый шаг пропускается, если его результат
уже есть. Если сессия Colab оборвалась — просто запустите ноутбук снова.

## 0. Настройка

In [1]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    token = userdata.get('GH_TOKEN')
    !git clone -q https://{token}@github.com/zhannaa0/kazakh-hybrid-nlp.git /content/project 2>/dev/null || git -C /content/project pull -q
    %cd /content/project
    !pip install -q -r requirements.txt
    PROJECT, THESIS = Path('/content/project'), Path('/content/drive/MyDrive/thesis')
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    THESIS = Path(os.environ.get('THESIS_DIR', PROJECT.parent / 'thesis'))
sys.path.insert(0, str(PROJECT))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите T4 для шага 2')
PREDS = THESIS / 'outputs' / 'preds'
SEEDS = [13, 42, 2026]

Mounted at /content/drive
/content/project
GPU: Tesla T4


In [2]:
import json
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Rule-based метод
Морфологический анализатор (аффиксы + словари) для POS и FEATS, шаблоны и газеттир для NER.
Работает на CPU за ~15 секунд.

In [3]:
from src.rules import run as rules
if not (PREDS / 'test' / 'ner_rule.jsonl').exists():
    rules.run(THESIS)
!python -m src.eval.evaluate {PREDS}/dev --out {THESIS}/outputs/results_dev

  dev: /content/drive/MyDrive/thesis/outputs/preds/dev
  test: /content/drive/MyDrive/thesis/outputs/preds/test
method       metric   rule
task                      
ner              f1  70.62
pos        accuracy  86.89
morph   exact_match  82.64

Сохранено в /content/drive/MyDrive/thesis/outputs/results_dev/: summary_3x3.csv, metrics/


## 2. Нейросеть
Один трансформер + головы задач. **UD** (POS + FEATS одной моделью): ~2–4 мин на seed на T4.
**NER** (KazNERD, 90 тыс. предложений): ~40 мин на seed — можно запускать по одному seed за сессию.

`MODEL` — открытое решение: сначала сравните два «хребта» на UD с одним seed (ячейка ниже),
затем зафиксируйте выбор в decision log.

In [4]:
from src.neural.train import train

MODEL = 'xlm-roberta-base'          # или 'kz-transformers/kaz-roberta-conversational'

def done(task, seed):
    name = 'ner' if task == 'ner' else 'morph'
    return (PREDS / 'test' / f'{name}_neural_seed{seed}.jsonl').exists()

In [5]:
# (необязательно) выбор «хребта»: оба на UD, seed 13, сравнение по dev
# for m in ['xlm-roberta-base', 'kz-transformers/kaz-roberta-conversational']:
#     log = train(THESIS, 'ud', m, seed=13)
#     print(m, log['best_dev_score'])

In [6]:
for seed in SEEDS:                  # POS + морфология
    if not done('ud', seed):
        print(f'UD, seed {seed}'); train(THESIS, 'ud', MODEL, seed=seed)

UD, seed 13


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (6 outputs left out)

  epoch  1  loss 23.4485  dev 0.3413
  epoch  2  loss 9.1837  dev 0.4760
  epoch  3  loss 6.0145  dev 0.6279
  epoch  4  loss 3.8532  dev 0.7812
  epoch  5  loss 2.4988  dev 0.8594
  epoch  6  loss 1.7747  dev 0.8684
  epoch  7  loss 1.3169  dev 0.9005
  epoch  8  loss 0.9899  dev 0.9028
  epoch  9  loss 0.7931  dev 0.9114
  epoch 10  loss 0.6547  dev 0.9227
  epoch 11  loss 0.5371  dev 0.9200
  epoch 12  loss 0.4432  dev 0.9218
  epoch 13  loss 0.3641  dev 0.9290
  epoch 14  loss 0.3185  dev 0.9222
  epoch 15  loss 0.2636  dev 0.9263
  epoch 16  loss 0.2153  dev 0.9286
  epoch 17  loss 0.1954  dev 0.9263
  epoch 18  loss 0.1671  dev 0.9277
  epoch 19  loss 0.1514  dev 0.9281
  epoch 20  loss 0.1256  dev 0.9281
  лучший dev 0.9290; предсказания в /content/drive/MyDrive/thesis/outputs/preds
UD, seed 2026


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/project/src/neural/train.py:205: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  epoch  1  loss 23.4156  dev 0.3422
  epoch  2  loss 8.8852  dev 0.4860
  epoch  3  loss 5.6414  dev 0.6609
  epoch  4  loss 3.5655  dev 0.7839
  epoch  5  loss 2.3320  dev 0.8495
  epoch  6  loss 1.6452  dev 0.8829
  epoch  7  loss 1.2676  dev 0.8983
  epoch  8  loss 0.9754  dev 0.9005
  epoch  9  loss 0.7760  dev 0.9132
  epoch 10  loss 0.6419  dev 0.9213
  epoch 11  loss 0.5203  dev 0.9186
  epoch 12  loss 0.4450  dev 0.9222
  epoch 13  loss 0.3721  dev 0.9231
  epoch 14  loss 0.3050  dev 0.9286
  epoch 15  loss 0.2568  dev 0.9227
  epoch 16  loss 0.2183  dev 0.9277
  epoch 17  loss 0.1850  dev 0.9286
  epoch 18  loss 0.1628  dev 0.9322
  epoch 19  loss 0.1470  dev 0.9317
  epoch 20  loss 0.1298  dev 0.9372
  epoch 21  loss 0.1053  dev 0.9340
  epoch 22  loss 0.0902  dev 0.9381
  epoch 23  loss 0.0792  dev 0.9358
  epoch 24  loss 0.0712  dev 0.9349
  epoch 25  loss 0.0634  dev 0.9335
  epoch 26  loss 0.0660  dev 0.9344
  epoch 27  loss 0.0526  dev 0.9358
  epoch 28  loss 0.0449  de

In [7]:
for seed in SEEDS:                  # NER — долго; при обрыве сессии запустите ячейку снова
    if not done('ner', seed):
        print(f'NER, seed {seed}'); train(THESIS, 'ner', MODEL, seed=seed)

NER, seed 13


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/project/src/neural/train.py:205: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  epoch  1  loss 0.2406  dev 0.9469
  epoch  2  loss 0.0387  dev 0.9605
  epoch  3  loss 0.0179  dev 0.9647
  лучший dev 0.9647; предсказания в /content/drive/MyDrive/thesis/outputs/preds
NER, seed 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch  1  loss 0.2354  dev 0.9489
  epoch  2  loss 0.0382  dev 0.9568
  epoch  3  loss 0.0180  dev 0.9634
  лучший dev 0.9634; предсказания в /content/drive/MyDrive/thesis/outputs/preds
NER, seed 2026


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch  1  loss 0.2300  dev 0.9440
  epoch  2  loss 0.0384  dev 0.9557
  epoch  3  loss 0.0179  dev 0.9637
  лучший dev 0.9637; предсказания в /content/drive/MyDrive/thesis/outputs/preds


## 3. Гибрид
Нейросеть → коррекция правилами. Параметры (λ для POS/FEATS; τ_p, γ, τ_g для NER) подбираются
на dev отдельно для каждого seed и применяются к test. Абляция — в `outputs/hybrid/params_seed*.json`.

In [8]:
from src.hybrid import run as hybrid
for seed in SEEDS:
    hybrid.run(THESIS, seed)

  ner: dev rule 0.7062 | neural 0.9647 | hybrid 0.9622  params {'tau_p': 0.95, 'gamma': 0.6, 'tau_g': 2.0}
  pos: dev rule 0.8689 | neural 0.9539 | hybrid 0.9548  params {'lambda': 0.1}
  morph: dev rule 0.8264 | neural 0.9268 | hybrid 0.9114  params {'lambda': 0.0}
  ner: dev rule 0.7062 | neural 0.9634 | hybrid 0.9608  params {'tau_p': 0.95, 'gamma': 0.6, 'tau_g': 2.0}
  pos: dev rule 0.8689 | neural 0.9430 | hybrid 0.9448  params {'lambda': 0.25}
  morph: dev rule 0.8264 | neural 0.9150 | hybrid 0.9114  params {'lambda': 0.1}
  ner: dev rule 0.7062 | neural 0.9637 | hybrid 0.9615  params {'tau_p': 0.95, 'gamma': 0.95, 'tau_g': 2.0}
  pos: dev rule 0.8689 | neural 0.9494 | hybrid 0.9494  params {'lambda': 0.0}
  morph: dev rule 0.8264 | neural 0.9313 | hybrid 0.9168  params {'lambda': 0.25}


In [9]:
p = json.load(open(THESIS / 'outputs' / 'hybrid' / f'params_seed{SEEDS[0]}.json'))
for task, r in p.items():
    display(Markdown(f"**{task}** — параметры {r['params']}"))
    display(pd.Series({'rule': r['dev_rule'], 'neural': r['dev_neural'], 'hybrid': r['dev_hybrid'],
                       **r['ablation_dev']}, name='dev').to_frame())

**ner** — параметры {'tau_p': 0.95, 'gamma': 0.6, 'tau_g': 2.0}

,dev
rule,0.7062
neural,0.9647
hybrid,0.9622
only patterns,0.9622
only gazetteer,0.9622
only IOB2 repair,0.9622


**pos** — параметры {'lambda': 0.1}

,dev
rule,0.8689
neural,0.9539
hybrid,0.9548
only hard constraints (λ=0),0.9539


**morph** — параметры {'lambda': 0.0}

,dev
rule,0.8264
neural,0.9268
hybrid,0.9114
only hard constraints (λ=0),0.9114


## 4. Итоговое сравнение на test (методичка 7.7, 8.5, 8.7)
Таблица 3×3 (среднее ± std по seed), все попарные сравнения с 95% ДИ и поправкой Холма.

In [10]:
RES = THESIS / 'outputs' / 'results_test'
!python -m src.eval.evaluate {PREDS}/test --out {RES} --significance

method       metric   rule        neural        hybrid
task                                                  
ner              f1  70.52  96.55 ± 0.22  96.31 ± 0.25
pos        accuracy  88.38  94.08 ± 1.10  94.26 ± 0.81
morph   exact_match  83.15  91.89 ± 0.59  90.84 ± 0.59

 task      metric      A      B  score_A  score_B  diff  ci95_low  ci95_high      p  runs_A  runs_B  p_holm  significant
  ner          f1 neural hybrid    96.55    96.31 -0.24     -0.29      -0.19 1.0000       3       3  1.0000        False
  ner          f1   rule hybrid    70.52    96.31 25.79     24.96      26.64 0.0000       1       3  0.0000         True
  ner          f1   rule neural    70.52    96.55 26.03     25.20      26.87 0.0000       1       3  0.0000         True
  pos    accuracy neural hybrid    94.08    94.26  0.18     -0.11       0.48 0.1211       3       3  0.3633        False
  pos    accuracy   rule hybrid    88.38    94.26  5.89      3.99       7.84 0.0000       1       3  0.0000         Tru

### Где помогает гибрид: известные и новые (OOV) слова

In [11]:
from src.data.prepare import load_processed
from src.eval.breakdown import by_oov, train_vocabulary
from src.eval.io import read_predictions

proc = THESIS / 'data' / 'processed'
vocab = {'ner': train_vocabulary(load_processed(proc, 'kaznerd', ('train',))['train']),
         'pos': train_vocabulary(load_processed(proc, 'udktb', ('train',))['train'])}
vocab['morph'] = vocab['pos']
rows = []
for task in ('ner', 'pos', 'morph'):
    for method in ('rule', f'neural_seed{SEEDS[0]}', f'hybrid_seed{SEEDS[0]}'):
        path = PREDS / 'test' / f'{task}_{method}.jsonl'
        if path.exists():
            t = by_oov(read_predictions(path), task, vocab[task])
            col = 'f1' if task == 'ner' else 'score'
            rows.append({'task': task, 'method': method.split('_')[0],
                         'in_vocab': t.loc['in_vocab', col], 'oov': t.loc['oov', col]})
pd.DataFrame(rows).pivot(index='method', columns='task', values=['in_vocab', 'oov'])

in_vocab                  oov              
task      morph    ner    pos  morph    ner    pos
method                                            
hybrid   0.9444 0.9715 0.9634 0.8548 0.8786 0.9247
neural   0.9526 0.9732 0.9621 0.8683 0.8794 0.9274
rule     0.9079 0.7206 0.9377 0.6801 0.4152 0.7769